# Notebook 06: Submission Pipeline Check

**Purpose:** Verify the end-to-end pipeline from raw test features to a valid submission file.

This notebook does NOT aim for a good score. It confirms:
1. Test features load correctly.
2. Predictions can be generated for all 424 targets.
3. The submission DataFrame has the correct shape and format.
4. Local evaluation against test labels (per-lag) works.

Run this notebook once before any actual submission to catch pipeline bugs early.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from src.data.load_data import (
    load_train, load_labels, load_test,
    load_target_pairs, load_test_labels, get_labels_for_lag,
)
from src.evaluation.metric import spearman_sharpe_score
from src.utils.paths import SUBMISSIONS_DIR, ensure_dirs

warnings.filterwarnings('ignore')
ensure_dirs()

print('Imports OK')

## 1. Load data

In [ ]:
train  = load_train()
labels = load_labels()
test   = load_test()
pairs  = load_target_pairs()

feature_cols_train = [c for c in train.columns if c != 'date_id']
feature_cols_test  = [c for c in test.columns  if c != 'date_id']

# Use only features common to both train and test
common_features = [c for c in feature_cols_train if c in feature_cols_test]
print(f'Train features : {len(feature_cols_train)}')
print(f'Test features  : {len(feature_cols_test)}')
print(f'Common features: {len(common_features)}')

## 2. Build pipeline (train on full training set)

In [ ]:
def build_lag_features(df, cols, lags=[1, 2, 3, 5, 10]):
    frames = []
    for lag in lags:
        shifted = df[cols].shift(lag)
        shifted.columns = [f'{c}_lag{lag}' for c in cols]
        frames.append(shifted)
    return pd.concat(frames, axis=1)

def impute_with_means(arr, means):
    out = arr.copy()
    idx = np.where(np.isnan(out))
    out[idx] = means[idx[1]]
    return out


# Lag features for train
train_lag = build_lag_features(train, common_features)
train_lag.insert(0, 'date_id', train['date_id'].values)

valid_rows = ~np.all(np.isnan(train_lag.drop(columns='date_id').values), axis=1)
X_all = train_lag.drop(columns='date_id').values[valid_rows]

col_means = np.nanmean(X_all, axis=0)
X_all_imp = impute_with_means(X_all, col_means)

scaler = StandardScaler()
X_all_sc = scaler.fit_transform(X_all_imp)

# Lag features for test (using the end of train + test, shift uses train history)
# Concatenate train + test for proper lagging, then slice test rows
combined = pd.concat([
    train[['date_id'] + common_features],
    test[['date_id'] + common_features],
], ignore_index=True)
combined_lag = build_lag_features(combined, common_features)
combined_lag.insert(0, 'date_id', combined['date_id'].values)

test_date_ids = test['date_id'].values
test_lag = combined_lag[combined_lag['date_id'].isin(test_date_ids)]
X_test = test_lag.drop(columns='date_id').values
X_test_imp = impute_with_means(X_test, col_means)
X_test_sc = scaler.transform(X_test_imp)

print(f'X_train: {X_all_sc.shape}   X_test: {X_test_sc.shape}')

## 3. Generate predictions for all 4 lags

In [ ]:
all_predictions = {}

train_date_ids = train_lag['date_id'].values[valid_rows]

for lag in [1, 2, 3, 4]:
    lag_labels  = get_labels_for_lag(labels, pairs, lag)
    target_cols = [c for c in lag_labels.columns if c != 'date_id']
    y_all = lag_labels.set_index('date_id')[target_cols].loc[train_date_ids].values
    y_all_imp = np.where(np.isnan(y_all), 0.0, y_all)

    model = Ridge(alpha=1.0)
    model.fit(X_all_sc, y_all_imp)

    preds = model.predict(X_test_sc)
    all_predictions[lag] = pd.DataFrame(
        preds,
        index=test_date_ids,
        columns=target_cols,
    )
    print(f'  Lag {lag}: predictions shape {preds.shape}')

## 4. Local evaluation against held-out test labels

In [ ]:
print('Local evaluation (Spearman-Sharpe on test labels):')

for lag in [1, 2, 3, 4]:
    test_labels = load_test_labels(lag)
    tl_cols = [c for c in test_labels.columns if c != 'date_id']
    tl_indexed = test_labels.set_index('date_id')[tl_cols]

    pred_cols = all_predictions[lag].columns.tolist()
    common_targets = [c for c in tl_cols if c in pred_cols]
    common_ids = [i for i in all_predictions[lag].index if i in tl_indexed.index]

    if not common_ids:
        print(f'  Lag {lag}: No overlapping date_ids between predictions and test labels')
        continue

    y_pred = all_predictions[lag].loc[common_ids, common_targets]
    y_true = tl_indexed.loc[common_ids, common_targets]

    score = spearman_sharpe_score(y_pred, y_true, nan_policy='ignore')
    print(f'  Lag {lag}: Spearman-Sharpe = {score:.4f}  '
          f'(date_ids {min(common_ids)}–{max(common_ids)}, {len(common_ids)} periods)')

## 5. Build and save submission file

In [ ]:
# Combine all lag predictions into one wide DataFrame
# (date_id × all 424 targets)
submission = pd.concat(
    [all_predictions[lag] for lag in [1, 2, 3, 4]],
    axis=1,
)
submission.index.name = 'date_id'
submission = submission.reset_index()

print(f'Submission shape: {submission.shape}')
print(f'Expected columns: 1 (date_id) + 424 targets = 425')
assert submission.shape[1] == 425, f'Unexpected column count: {submission.shape[1]}'

# Save
output_path = SUBMISSIONS_DIR / 'baseline_ridge_lag_features.csv'
submission.to_csv(output_path, index=False)
print(f'Saved to {output_path}')
submission.head()

## 6. Pipeline checklist

| Check | Status |
|-------|--------|
| Test features loaded | ✓ |
| Lag features built using train history | ✓ |
| Predictions for all 4 lags (424 targets) | ✓ |
| Local Spearman-Sharpe scores computed | ✓ |
| Submission file saved to submissions/ | ✓ |
| Submission shape = (134, 425) | Verify |

Fill in scores and verify shape after running.